# Initialisation

In [5]:
from src.backend.vector_store import build_vector_store
from sentence_transformers import SentenceTransformer
import pandas as pd

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from src.backend import config  # side effect: charge le .env + HF_TOKEN

# Import

In [2]:
df_chunks = pd.read_csv("../data/processed/parcoursup_2026_enriched_chunks.csv")  # ou ton fichier réel
df_sample = df_chunks.sample(10, random_state=0)

In [3]:
df_sample["chunk_text"].head()
type(df_sample["chunk_text"].iloc[0])
df_sample["chunk_text"].apply(type).value_counts()

chunk_text
<class 'str'>    10
Name: count, dtype: int64

# Test fonction vectorisation

In [4]:
df_vs = build_vector_store(df_sample)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15473.98it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
df_vs.head()

,chunk_id,chunk_index,chunk_text,type_formation,type_etablissement,commune,is_apprentissage,frais_scolarite,formation_selective,formation_ouverte_boursiers,embedding
5235,4398_0,0,Nom de la formation : Double licence - Langues...,"Licence sélective,Licence",Publics,Paris,0,NaN,1.0,1.0,"[0.0032532387, 0.06998689, -0.006568642, 0.027..."
3302,2658_0,0,Nom de la formation : BTS - Services - Gestion...,BTS - BTSA - BTSM,Publics,Montmorillon,0,NaN,1.0,1.0,"[0.039888863, 0.06658044, -0.011404911, 0.0260..."
6804,5752_0,0,Nom de la formation : Formation d'ingénieur Ba...,Formations des écoles d'ingénieurs,Publics,Limoges,0,"En 2025, 618 euros de droits de scolarité",1.0,1.0,"[0.005525817, 0.05455804, -0.0141268, 0.027466..."
7384,6162_0,0,Nom de la formation : Licence - Parcours d'Acc...,"Etudes de santé,Licence",Publics,Nîmes,0,178 euros.,0.0,1.0,"[0.016017511, 0.041464224, -0.04017601, 0.0381..."
22440,18554_0,0,Nom de la formation : BTS - Services - Managem...,BTS - BTSA - BTSM,Privés,La Tour-d'Aigues,1,NaN,1.0,1.0,"[0.0049913153, 0.053549267, -0.019196438, 0.02..."


In [6]:
df_vs["embedding"].iloc[:5]  # voir les 5 premiers coeffs

5235     [0.0032532387, 0.06998689, -0.006568642, 0.027...
3302     [0.039888863, 0.06658044, -0.011404911, 0.0260...
6804     [0.005525817, 0.05455804, -0.0141268, 0.027466...
7384     [0.016017511, 0.041464224, -0.04017601, 0.0381...
22440    [0.0049913153, 0.053549267, -0.019196438, 0.02...
Name: embedding, dtype: object

# Chargement du token HF + chargement modèle

In [6]:
from src.backend import config  # doit charger .env et HF_TOKEN
from sentence_transformers import SentenceTransformer
import os

print("HF_TOKEN vu par Python :", os.getenv("HF_TOKEN")[:8], "...")

model = SentenceTransformer("intfloat/multilingual-e5-base")
emb = model.encode(["test"], normalize_embeddings=True)
emb.shape

HF_TOKEN vu par Python : hf_DTtbl ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7011.00it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(1, 768)

# Tester la connexion avec pgvector

In [7]:
from src.backend.pgvector_store import get_pg_connection

conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())
cur.close()
conn.close()

(1,)
